In [2]:
%pip install requests beautifulsoup4 tqdm pillow selenium webdriver-manager

import os
import time
import requests
from tqdm import tqdm
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from PIL import Image
from io import BytesIO

Note: you may need to restart the kernel to use updated packages.


In [14]:
BASE_URL = "https://www.ikea.com/fr/fr/cat/chaises-de-salle-a-manger-25219/"  # à remplacer
SAVE_DIR = "dataset/temporary"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
}

os.makedirs(SAVE_DIR, exist_ok=True)

In [15]:
def get_best_src_from_srcset(srcset):
    # prend l'image la plus haute résolution
    sources = srcset.split(",")
    last = sources[-1].strip().split(" ")[0]
    return last

In [16]:
# Idée: scrapper les noms de produits puis récup direct l'image dans leur page

def extract_product_urls(page_url):
    response = requests.get(page_url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    urls = []

    product_links = soup.find_all("a", class_="plp-product__image-link")

    for link in product_links:
        href = link.get("href")
        if href:
            urls.append(href)

    return list(set(urls))

In [ ]:
# def extract_image_urls(page_url):
#     response = requests.get(page_url, headers=HEADERS)
#     soup = BeautifulSoup(response.text, "html.parser")

#     image_urls = []

#     # 1️⃣ cibler les liens produits
#     product_links = soup.find_all("a", class_="plp-product__image-link")

#     for link in product_links:
#         # 2️⃣ récupérer uniquement l'image principale (pas --alt)
#         img = link.find("img", class_="plp-product__image")

#         if img:
#             url = img.get("src")

#             if url:
#                 # 3️⃣ enlever la version low-res
#                 url = url.split("?")[0]
#                 image_urls.append(url)

#     print(f"{len(list(set(image_urls)))} image urls retrieved")
#     return list(set(image_urls))

In [17]:
CONFIG_IKEA = {
    "container_classes": ["plp-product__image-link"],
    "image_classes": ["plp-product__image"],
    "exclude_classes": ["--alt"],
    "high_res": True
}

CONFIG_ALINEA = {
    "container_classes": ["product-image-name-container"],
    "image_classes": ["product-detail-image"],
    "exclude_classes": [],
    "high_res": True
}

In [18]:
CONFIG_ALL = {
    "container_classes": ["plp-product__image-link", "product-image-name-container"],
    "image_classes": ["plp-product__image", "product-detail-image"],
    "exclude_classes": ["--alt"],
    "high_res": True
}

In [19]:
def extract_image_urls(page_url, config, headers=None):

    response = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    image_urls = []

    container_classes = config.get("container_classes", [])
    image_classes = config.get("image_classes", [])
    exclude_classes = config.get("exclude_classes", [])
    high_res = config.get("high_res", False)

    # 1️⃣ trouver les containers
    containers = []

    for class_name in container_classes:
        containers.extend(soup.find_all(class_=class_name))

    for container in containers:
        imgs = container.find_all("img")

        for img in imgs:
            classes = img.get("class", [])

            # 2️⃣ filtrer par classes images si fourni
            if image_classes:
                if not any(cls in classes for cls in image_classes):
                    continue

            # 3️⃣ exclure certaines classes
            if any(excl in " ".join(classes) for excl in exclude_classes):
                continue

            # 4️⃣ récupérer la meilleure URL possible
            url = None

            if img.get("data-src"):
                url = img.get("data-src")

            elif img.get("srcset"):
                srcset = img.get("srcset")
                sources = srcset.split(",")
                url = sources[-1].strip().split(" ")[0]

            elif img.get("src"):
                url = img.get("src")

            if not url:
                continue

            # 5️⃣ transformer en URL absolue
            url = urljoin(page_url, url)

            # 6️⃣ supprimer paramètres pour HD
            if high_res:
                url = url.split("?")[0]

            image_urls.append(url)

    return list(set(image_urls))

In [20]:
def download_image(url, save_path, min_size=128):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code != 200:
            print(f"Failed to download {url}: Status code {response.status_code}")
            return False
        
        img = Image.open(BytesIO(response.content)).convert("RGB")
        
        if min(img.size) < min_size:
            print(f"Image {url} is too small: {img.size}")
            return False
        
        img.save(save_path)
        return True
    
    except Exception as e:
        print(f"Error downloading image: {e}")
        return False

In [28]:
def scrape_dataset(start_page, num_pages=5):
    counter = 0
    
    for page in range(1, num_pages + 1):
        page_url = f"{start_page}?page={page}"
        print(f"Scraping {page_url}")
        
        image_urls = extract_image_urls(page_url, config=CONFIG_ALL, headers=HEADERS)
        
        for url in tqdm(image_urls):
            # On appelle l'image par le nom de son url pour pouvoir sourcer plus tard
            save_path = os.path.join(SAVE_DIR, f"chair_{os.path.basename(url.split('?')[0])}")
            
            if download_image(url, save_path):
                counter += 1
            
            time.sleep(0.1)  # important : éviter le spam
    
    print(f"Downloaded {counter} images.")

In [49]:
# Vider le dossier dataset temporaire

for filename in os.listdir(SAVE_DIR):
    file_path = os.path.join(SAVE_DIR, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)

In [48]:
# Eliminer les doublons

seen_hashes = set()

print("Before removing duplicates, number of images:", len(os.listdir(FINAL_DIR)))
FINAL_DIR = "dataset/verified"

for filename in os.listdir(FINAL_DIR):
    file_path = os.path.join(FINAL_DIR, filename)
    if os.path.isfile(file_path):
        try:
            img = Image.open(file_path)
            img_hash = hash(img.tobytes())
            if img_hash in seen_hashes:
                os.remove(file_path)
            else:
                seen_hashes.add(img_hash)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

print("After removing duplicates, number of images:", len(os.listdir(FINAL_DIR)))

Before removing duplicates, number of images: 147
After removing duplicates, number of images: 147


In [47]:
# Transférer les images dans le dossier de sauvegarde final
FINAL_DIR = "dataset/verified"
os.makedirs(FINAL_DIR, exist_ok=True)
for filename in os.listdir(SAVE_DIR):
    src_path = os.path.join(SAVE_DIR, filename)
    dst_path = os.path.join(FINAL_DIR, filename)
    if os.path.isfile(src_path):
        if not os.path.isfile(dst_path):
            os.rename(src_path, dst_path)

In [ ]:
URLS_TRAITEES_1_page = {
    "https://www.ikea.com/fr/fr/cat/chaises-700676/": "chaises",
    "https://www.ikea.com/fr/fr/cat/chaises-de-salle-a-manger-25219/": "chaises_de_salle_a_manger",
    "https://www.ikea.com/fr/fr/cat/chaises-pliantes-25222/": "chaises_pliantes",
    "https://www.ikea.com/fr/fr/cat/chaises-de-bar-20864/": "chaises_de_bar",
    "https://www.ikea.com/fr/fr/cat/chaises-de-cafe-et-restaurant-19144/": "chaises_de_cafe_et_restaurant",
    "https://www.ikea.com/fr/fr/cat/fauteuils-fu006/": "fauteuils",
    "https://www.ikea.com/fr/fr/cat/fauteuils-en-tissu-10687/": "fauteuils_en_tissu",
    "https://www.ikea.com/fr/fr/cat/fauteuils-en-cuir-10696/": "fauteuils_en_cuir",
    "https://www.ikea.com/fr/fr/cat/fauteuils-en-rotin-20907/": "fauteuils_en_rotin",
    "https://www.ikea.com/fr/fr/cat/fauteuils-enfant-20483/": "fauteuils_enfant",
    "https://www.ikea.com/fr/fr/cat/fauteuils-club-53257/": "fauteuils_club",
    "https://www.ikea.com/fr/fr/cat/chauffeuses-convertibles-16296/": "chauffeuses_convertibles",
    "https://www.ikea.com/fr/fr/cat/chaises-de-table-25220/": "chaises_de_table",
    "https://www.ikea.com/fr/fr/cat/chaises-rembourrees-25221/": "chaises_rembourrees"
}

In [ ]:
URLS = {
    "https://www.alinea.com/fr-fr/chaises/": "chaises",
}

for url, category in URLS.items():
    print(f"{"="*50} Scraping category: {category} {"="*50}")
    scrape_dataset(url, num_pages=5)

================================================== Scraping category: chaises ==================================================
Scraping https://www.alinea.com/fr-fr/chaises/?page=1


0it [00:00, ?it/s]


Scraping https://www.alinea.com/fr-fr/chaises/?page=2


0it [00:00, ?it/s]


Scraping https://www.alinea.com/fr-fr/chaises/?page=3


...

Scraping https://www.ikea.com/fr/fr/cat/chauffeuses-convertibles-16296/?page=4


100%|██████████| 6/6 [00:05<00:00,  1.10it/s]


Scraping https://www.ikea.com/fr/fr/cat/chauffeuses-convertibles-16296/?page=5


100%|██████████| 6/6 [00:05<00:00,  1.14it/s]

Downloaded 30 images.
